In [1]:
import sys
sys.path.append("..")

import h5py
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from utils.das_dataset import DasDataset
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from utils.noise_scheduler import LinearNoiseScheduler 
from IPython.display import clear_output

In [2]:
das_data = Path("/datasets/training/earthquake/share/SISSLE_south30")
train_files_noise = [das_data/"train_1000.h5"]#sorted(das_data.glob("train_*.h5"))
test_files_noise = [das_data/"test_1000.h5"]#sorted(das_data.glob("test_*.h5"))

device= "cuda" if torch.cuda.is_available() else "cpu"

train_set_noise = DasDataset(train_files_noise)
test_set_noise = DasDataset(test_files_noise)

In [3]:
tail_scores_path = Path ("data/noise_unshifted_tail_scores.npy")
tail_scores_path.parent.mkdir(exist_ok = True)

if tail_scores_path.exists():
    tail_scores = np.load(tail_scores_path)

else:
    tail_scores = []
    for file, file_path in enumerate (train_files_noise):
        print(f"processing file {file+1} / {len(train_files_noise)} :") 
        f = h5py.File(file_path, "r")
        patches = f["patches"]
        num_patches = patches.shape[0]

        for index in tqdm(range(num_patches)):

            patch = np.array(patches[index])
            tail_score = np.percentile(np.abs(patch),99.9)
            tail_scores.append(tail_score)

    tail_scores = np.asarray(tail_scores, dtype = np.float32)
    np.save(tail_scores_path, tail_scores)

    print("Tail scores saved")
print("Tail scores shape:", tail_scores.shape)            

processing file 1 / 1 :


  0%|          | 0/16384 [00:00<?, ?it/s]

Tail scores saved
Tail scores shape: (16384,)


In [4]:
upper_threshold = np.percentile(tail_scores,80)
extreme_threshold = np.percentile(tail_scores,99)

upper_indices = np.where((tail_scores >= upper_threshold) & (tail_scores < extreme_threshold))[0]
extreme_indices = np.where(tail_scores >= extreme_threshold)[0]
all_indices = np.arange(len(tail_scores))

print("all patches: ", len(all_indices))
print("upper tail patches: ", len(upper_indices))
print("extreme tail patches: ", len(extreme_indices))

all patches:  16384
upper tail patches:  3113
extreme tail patches:  164


In [5]:
train_files_earthquake = [das_data/"train_1001.h5"]
test_files_earthquake = [das_data/"test_1001.h5"]

device= "cuda" if torch.cuda.is_available() else "cpu"

train_set_earthquake = DasDataset(train_files_earthquake)
test_set_earthquake = DasDataset(test_files_earthquake)

In [6]:
earthquake_scores_path = Path ("data/earthquake_unshifted_tail_scores.npy")
earthquake_scores_path.parent.mkdir(exist_ok = True)

if earthquake_scores_path.exists():
    earthquake_scores = np.load(earthquake_scores_path)

else:
    earthquake_scores = []
    for file, file_path in enumerate (train_files_earthquake):
        print(f"processing file {file+1} / {len(train_files_earthquake)} :") 
        f = h5py.File(file_path, "r")
        patches = f["patches"]
        num_patches = patches.shape[0]

        for index in tqdm(range(num_patches)):

            patch = np.array(patches[index])
            earthquake_score = np.percentile(np.abs(patch),99.9)
            earthquake_scores.append(earthquake_score)

    earthquake_scores = np.asarray(earthquake_scores, dtype = np.float32)
    np.save(earthquake_scores_path, earthquake_scores)

    print("earthquake scores saved")
print("earthquake scores shape:", earthquake_scores.shape)

processing file 1 / 1 :


  0%|          | 0/16384 [00:00<?, ?it/s]

earthquake scores saved
earthquake scores shape: (16384,)


In [7]:
medium_threshold = np.percentile(earthquake_scores,50)
strong_threshold = np.percentile(earthquake_scores,85)
extreme_threshold = np.percentile(earthquake_scores,98)

weak_indices = np.where((earthquake_scores < medium_threshold) )[0]
medium_indices = np.where((earthquake_scores >= medium_threshold) & (earthquake_scores < strong_threshold))[0]
strong_indices = np.where((earthquake_scores >= strong_threshold) & (earthquake_scores < extreme_threshold))[0]
extreme_indices = np.where(earthquake_scores >= extreme_threshold)[0]


print("weak patches: ", len(weak_indices))
print("medium patches: ", len(medium_indices))
print("strong patches: ", len(strong_indices))
print("extreme patches: ", len(extreme_indices))

weak patches:  8192
medium patches:  5734
strong patches:  2130
extreme patches:  328
